# A.2: The cost of the wrong template

Score the same reference answers under SmolLM2-135M-Instruct with its own template and with three other formats.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 One conversation, three chat templates
import torch

import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

conversation = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2 + 3?"},
    {"role": "assistant", "content": "2 + 3 = 5."},
]

names = ["HuggingFaceTB/SmolLM2-135M-Instruct", "Qwen/Qwen2.5-0.5B-Instruct",
         "TinyLlama/TinyLlama-1.1B-Chat-v1.0"]

for name in names:
    t = AutoTokenizer.from_pretrained(name)
    text = t.apply_chat_template(conversation, tokenize=False)
    n_total = len(t(text, add_special_tokens=False).input_ids)
    n_content = sum(len(t(m["content"], add_special_tokens=False).input_ids) for m in conversation)
    print(f"== {name}  ({n_total} tokens, {n_content} of them message content)")
    print(text)


In [ ]:
tok = AutoTokenizer.from_pretrained(names[0])
model = AutoModelForCausalLM.from_pretrained(names[0], dtype=torch.float32).eval()
qa = [("What is the capital of Japan?", "The capital of Japan is Tokyo."),
      ("How many days are in a week?", "There are seven days in a week."),
      ("What color is the sky on a clear day?", "The sky is blue on a clear day."),
      ("What is the opposite of hot?", "The opposite of hot is cold."),
      ("Name a fruit that is yellow.", "A banana is a yellow fruit."),
      ("What do bees make?", "Bees make honey."),
      ("How many legs does a spider have?", "A spider has eight legs."),
      ("What is frozen water called?", "Frozen water is called ice.")]

def own_template(q):
    return tok.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                   add_generation_prompt=True)
formats = {
    "own template": own_template,
    "own template, no system turn": lambda q: f"<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n",
    "Llama-2 style [INST]": lambda q: f"[INST] {q} [/INST] ",
    "plain User:/Assistant:": lambda q: f"User: {q}\nAssistant: ",
}

def response_nll(prompt, answer):
    p = tok(prompt, add_special_tokens=False).input_ids
    a = tok(answer, add_special_tokens=False).input_ids
    x = torch.tensor([p + a])
    with torch.no_grad():
        logits = model(x).logits[0, len(p) - 1:-1]
    return F.cross_entropy(logits, x[0, len(p):]).item()

for label, fmt in formats.items():
    nll = sum(response_nll(fmt(q), a) for q, a in qa) / len(qa)
    print(f"{label:30s} mean loss per answer token {nll:.3f}")


**Expected output**

Output:

```text
own template                   mean loss per answer token 0.795
own template, no system turn   mean loss per answer token 0.741
Llama-2 style [INST]           mean loss per answer token 2.568
plain User:/Assistant:         mean loss per answer token 2.262
```


In [ ]:
q = "What is the opposite of hot?"
for label in ["own template", "plain User:/Assistant:"]:
    ids = tok(formats[label](q), return_tensors="pt", add_special_tokens=False).input_ids
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=30, do_sample=False, pad_token_id=tok.eos_token_id)
    print(f"{label}: {tok.decode(out[0, ids.shape[1]:])!r}")


**Expected output**

Output:

```text
own template: 'The opposite of hot is cold. This is because heat is a form of energy that is typically associated with warmth, comfort, and activity. When we'
plain User:/Assistant:: '\nThe opposite of hot is cold.\n\nSo, the correct answer is:\n\nThe opposite of hot is cold.<|im_end|>'
```
